In [ ]:
!pip install torch-geometric


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.2 MB/s eta 0:00:00


In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)

2.10.0+cpu
None


In [ ]:
!pip install torch-geometric



In [ ]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from scipy.stats import pearsonr
from scipy.stats import pearsonr

import random

In [ ]:
ppi = pd.read_csv("/content/kidney_PPI_final.csv")
labels = pd.read_csv("/content/labels.csv")
expressions = pd.read_csv("/content/expressions.csv")
mutations = pd.read_csv("/content/mutations.csv")

In [ ]:
ppi.shape

(1528076, 4)

In [ ]:
ppi_genes = pd.concat([ppi['A'], ppi['B']]).unique()
gene_to_idx = {gene: i for i, gene in enumerate(ppi_genes)}

In [ ]:
edge_src = ppi['A'].map(gene_to_idx).values
edge_dst = ppi['B'].map(gene_to_idx).values

edge_index = torch.tensor(
    np.array([np.concatenate([edge_src, edge_dst]),
              np.concatenate([edge_dst, edge_src])]),
    dtype=torch.long
)


edge_features = torch.tensor(ppi[['combined_score', 'coexpression']].values, dtype=torch.float)
edge_attr = torch.cat([edge_features, edge_features], dim=0)
edge_attr = (edge_attr - edge_attr.mean(dim=0)) / edge_attr.std(dim=0)


In [ ]:
expr_no_id = expressions.drop('ModelID', axis=1)
mut_no_id = mutations.drop('ModelID', axis=1)
mutations_aligned = mut_no_id.reindex(columns=ppi_genes, fill_value=0)
labels_no_id = labels.drop('ModelID', axis=1)
labels_reindexed = labels_no_id.reindex(columns=ppi_genes)

In [ ]:
graph_list = []
for i in range(32):
    expr_row = expr_no_id[ppi_genes].iloc[i].values
    mut_row = mutations_aligned.iloc[i].values
    x = torch.tensor(np.column_stack([expr_row, mut_row]), dtype=torch.float)

    y_raw = labels_reindexed.iloc[i]
    cell_mask = torch.tensor((~y_raw.isna().values) & np.isin(ppi_genes, labels_no_id.columns), dtype=torch.bool)
    y = torch.tensor(y_raw.fillna(0).values, dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, mask=cell_mask)
    graph_list.append(data)


In [ ]:
random.seed(12)
random.shuffle(graph_list)

train_data = graph_list[:26]
test_data = graph_list[26:]

from torch_geometric.loader import DataLoader
train_loader = DataLoader(train_data, batch_size=1, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1)

In [ ]:
import torch.nn as nn
from torch_geometric.nn import GATConv

In [ ]:
class KidneyGAT(nn.Module):
    def __init__(self):
        super().__init__()
        self.expr_encoder = nn.Linear(1, 32)
        self.mut_encoder = nn.Linear(1, 32)
        self.conv1 = GATConv(64, 64, heads=4, edge_dim=2)
        self.conv2 = GATConv(64 * 4, 64, heads=4, edge_dim=2, concat=False)
        self.relu = nn.LeakyReLU()
        self.dropout = nn.Dropout(0.3)
        self.out = nn.Linear(64, 1)

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        expr = self.expr_encoder(x[:, 0:1])
        mut = self.mut_encoder(x[:, 1:2])
        x = torch.cat([expr, mut], dim=1)

        x = self.conv1(x, edge_index, edge_attr)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index, edge_attr)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.out(x)
        return x.squeeze()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = KidneyGAT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
loss_fn = nn.MSELoss()

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=12)
all_attn = []
fold_pearsons = []

for fold, (train_idx, test_idx) in enumerate(kf.split(graph_list)):
    train_data = [graph_list[i] for i in train_idx]
    test_data = [graph_list[i] for i in test_idx]
    train_loader = DataLoader(train_data, batch_size=1, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=1)

    
    model = KidneyGAT().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    loss_fn = nn.MSELoss()

    # Train
    for epoch in range(200):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            pred = model(batch)
            loss = loss_fn(pred[batch.mask], batch.y[batch.mask])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # for evaluating + extracting attention
    model.eval()
    fold_corrs = []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)

            # Pearson
            pred = model(batch)
            p = pred[batch.mask].cpu().numpy()
            a = batch.y[batch.mask].cpu().numpy()
            r, _ = pearsonr(p, a)
            fold_corrs.append(r)

            # Attention extraction
            expr = model.expr_encoder(batch.x[:, 0:1])
            mut = model.mut_encoder(batch.x[:, 1:2])
            x = torch.cat([expr, mut], dim=1)
            x, _ = model.conv1(x, batch.edge_index, batch.edge_attr, return_attention_weights=True)
            x = model.relu(x)
            x = model.dropout(x)
            x, (edge_idx, scores) = model.conv2(x, batch.edge_index, batch.edge_attr, return_attention_weights=True)

            avg_scores = scores.mean(dim=1).cpu().numpy()
            all_attn.append(avg_scores)

    fold_r = np.mean(fold_corrs)
    fold_pearsons.append(fold_r)
    print(f"Fold {fold+1}: Pearson r = {fold_r:.4f}")

print(f"\nMean Pearson r across 5 folds: {np.mean(fold_pearsons):.4f} ± {np.std(fold_pearsons):.4f}")


Fold 1: Pearson r = 0.3972
Fold 2: Pearson r = 0.5139
Fold 3: Pearson r = 0.5491
Fold 4: Pearson r = 0.5704
Fold 5: Pearson r = 0.5237

Mean Pearson r across 5 folds: 0.5108 ± 0.0602


In [ ]:
# for edge indices from one sample
sample = graph_list[0].to(device)
with torch.no_grad():
    expr = model.expr_encoder(sample.x[:, 0:1])
    mut = model.mut_encoder(sample.x[:, 1:2])
    x = torch.cat([expr, mut], dim=1)
    x, _ = model.conv1(x, sample.edge_index, sample.edge_attr, return_attention_weights=True)
    x = model.relu(x)
    x = model.dropout(x)
    x, (edge_idx, _) = model.conv2(x, sample.edge_index, sample.edge_attr, return_attention_weights=True)

src = edge_idx[0].cpu().numpy()
dst = edge_idx[1].cpu().numpy()

# Average attention across all folds
mean_attn = np.mean(all_attn, axis=0)

# Filtering  self-loops
mask = src != dst
src_filtered = src[mask]
dst_filtered = dst[mask]
attn_filtered = mean_attn[mask]

idx_to_gene = {i: gene for gene, i in gene_to_idx.items()}

# Top 20 interactions
top_indices = np.argsort(attn_filtered)[-20:][::-1]
print("Top 20 gene interactions (averaged across all folds):\n")
for rank, i in enumerate(top_indices, 1):
    g1 = idx_to_gene[src_filtered[i]]
    g2 = idx_to_gene[dst_filtered[i]]
    print(f"{rank:2d}. {g1:10s} <-> {g2:10s}  attn: {attn_filtered[i]:.4f}")


Top 20 gene interactions (averaged across all folds):

 1. PSMD14     <-> CDK2AP1     attn: 0.6759
 2. AZIN1      <-> OAZ2        attn: 0.6626
 3. EMSY       <-> LIX1        attn: 0.6545
 4. LYPD4      <-> GASK1A      attn: 0.6532
 5. PSMA7      <-> ZFAND5      attn: 0.6525
 6. GOLGA6L9   <-> STK25       attn: 0.6489
 7. USP9Y      <-> EIF1AY      attn: 0.6430
 8. BAG4       <-> KRTAP13-4   attn: 0.6375
 9. SYT6       <-> SYT10       attn: 0.6373
10. FEM1C      <-> OR51B2      attn: 0.6341
11. SDCBP      <-> SSNA1       attn: 0.6336
12. POLR3D     <-> DCAF8L2     attn: 0.6329
13. TYW5       <-> POTEF       attn: 0.6276
14. RIGI       <-> SHISAL2A    attn: 0.6275
15. DHRS12     <-> IGLL5       attn: 0.6263
16. NFIB       <-> ZNF536      attn: 0.6257
17. RIGI       <-> C17orf99    attn: 0.6249
18. DCPS       <-> HS3ST6      attn: 0.6226
19. GAS7       <-> NWD1        attn: 0.6222
20. MEX3A      <-> INMT        attn: 0.6220


In [ ]:
known_cancer_genes = ['VHL', 'TP53', 'EGFR', 'PTEN', 'MTOR', 'MYC', 'KRAS', 'BRCA1', 'PIK3CA', 'BRAF']

for gene in known_cancer_genes:
    if gene not in gene_to_idx:
        print(f"{gene} — not in PPI network")
        continue

    idx = gene_to_idx[gene]
    gene_mask = (src_filtered == idx) | (dst_filtered == idx)
    gene_attn = attn_filtered[gene_mask]
    gene_src = src_filtered[gene_mask]
    gene_dst = dst_filtered[gene_mask]

    if len(gene_attn) == 0:
        print(f"{gene} — no edges found")
        continue

    top5 = np.argsort(gene_attn)[-5:][::-1]

    print(f"\n{gene} — {len(gene_attn)} edges, max attn: {gene_attn.max():.4f}")
    for i in top5:
        partner = idx_to_gene[gene_dst[i]] if gene_src[i] == idx else idx_to_gene[gene_src[i]]
        print(f"  <-> {partner:12s}  attn: {gene_attn[i]:.4f}")



VHL — 782 edges, max attn: 0.2834
  <-> ASTL          attn: 0.2834
  <-> NSUN7         attn: 0.2831
  <-> PKD1L3        attn: 0.2784
  <-> OR6K3         attn: 0.2671
  <-> GPLD1         attn: 0.2660

TP53 — 3317 edges, max attn: 0.5676
  <-> ZNF679        attn: 0.5676
  <-> FAM217A       attn: 0.5624
  <-> GASK1B        attn: 0.5086
  <-> TDRD12        attn: 0.3321
  <-> SPATA6L       attn: 0.3260

EGFR — 2821 edges, max attn: 0.5763
  <-> GPR149        attn: 0.5763
  <-> ZCCHC18       attn: 0.5076
  <-> CCDC60        attn: 0.3945
  <-> PAGE3         attn: 0.3633
  <-> CES4A         attn: 0.3082

PTEN — 1188 edges, max attn: 0.4908
  <-> QRFPR         attn: 0.4908
  <-> CMTM1         attn: 0.2265
  <-> ECSCR         attn: 0.1780
  <-> PDZD9         attn: 0.1632
  <-> P2RY11        attn: 0.1598

MTOR — 594 edges, max attn: 0.0827
  <-> ARSL          attn: 0.0827
  <-> KCNJ4         attn: 0.0678
  <-> PRR5L         attn: 0.0463
  <-> NPY2R         attn: 0.0419
  <-> TRMT61A       attn: 